# Advanced RAG: Hybrid Search & Reranking

## Hybrid Search: Dense + Sparse Retrieval

Hybrid search combines dense vector similarity (semantic) with sparse BM25 (lexical) retrieval. Dense retrieval captures semantic meaning; sparse retrieval excels at exact keyword matching. Combining both: $\text{score} = \alpha \cdot \text{dense\_score} + (1-\alpha) \cdot \text{sparse\_score}$ provides robust retrieval across different query types.

```python title="example1.py"
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Sample documents
documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with multiple layers.",
    "Natural language processing focuses on text understanding.",
    "Computer vision processes and analyzes images."
]

# Split documents
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = splitter.split_text("\n".join(documents))

# Dense retriever (semantic)
embeddings = HuggingFaceEmbeddings(model_name="paraphrase-MiniLM-L6-v2")
dense_retriever = FAISS.from_texts(chunks, embeddings).as_retriever(k=3)

# Sparse retriever (lexical)
sparse_retriever = BM25Retriever.from_texts(chunks)

# Ensemble retriever (hybrid)
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.6, 0.4]  # 60% dense, 40% sparse
)

# Retrieve documents
query = "neural networks and deep learning"
results = hybrid_retriever.get_relevant_documents(query)
for doc in results:
    print(f"- {doc.page_content[:80]}...")
```

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/rag-systems/mod-26.ipynb)

```
- Deep learning uses neural networks with multiple layers.
- Machine learning is a subset of artificial intelligence.
- Natural language processing focuses on text understanding.
```

## Cross-Encoder Reranking

Cross-encoders score query-document pairs directly, providing more accurate relevance scores than embedding similarity. A cross-encoder takes $[\text{query}, \text{document}]$ as input and outputs a relevance score: $\text{score} = \text{CrossEncoder}([\text{query}, \text{document}])$. This is more expensive than embedding similarity but highly accurate for reranking top-k results.

```python title="example2.py"
from sentence_transformers import CrossEncoder
from langchain.retrievers import BM25Retriever
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Sample documents
documents = [
    "Machine learning algorithms learn patterns from data.",
    "Deep learning uses neural networks with multiple layers.",
    "Supervised learning requires labeled training data.",
    "Unsupervised learning finds patterns without labels.",
    "Reinforcement learning learns through trial and error."
]

# Initial retrieval (BM25)
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = splitter.split_text("\n".join(documents))
retriever = BM25Retriever.from_texts(chunks)

# Retrieve top-10 candidates
query = "learning algorithms with neural networks"
candidates = retriever.get_relevant_documents(query)

# Rerank with cross-encoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
pairs = [[query, doc.page_content] for doc in candidates]
scores = cross_encoder.predict(pairs)

# Sort by cross-encoder score
reranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)

print("Top reranked results:")
for score, doc in reranked[:3]:
    print(f"Score: {score:.3f} - {doc.page_content[:60]}...")
```

> **💡 Tip:** Use cross-encoders for final reranking of top-k results (k=10-50). For initial retrieval, use dense or hybrid search for speed.

## Multi-Stage Retrieval Pipeline

Production RAG systems use multi-stage pipelines: (1) fast initial retrieval (dense/sparse), (2) candidate reranking (cross-encoder), (3) context expansion (adding surrounding chunks), (4) LLM generation.

```python title="example3.py"
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder

# Setup retrievers
embeddings = HuggingFaceEmbeddings(model_name="paraphrase-MiniLM-L6-v2")
documents = ["Machine learning...", "Deep learning..."]  # Your docs
dense_retriever = FAISS.from_texts(documents, embeddings).as_retriever(k=10)
sparse_retriever = BM25Retriever.from_texts(documents)
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.6, 0.4]
)

# Multi-stage pipeline
class MultiStageRetriever:
    def __init__(self, hybrid_retriever, cross_encoder_model):
        self.hybrid = hybrid_retriever
        self.cross_encoder = CrossEncoder(cross_encoder_model)
    
    def retrieve(self, query, k_final=3):
        # Stage 1: Hybrid retrieval (top-10)
        candidates = self.hybrid.get_relevant_documents(query)[:10]
        
        # Stage 2: Cross-encoder reranking
        pairs = [[query, doc.page_content] for doc in candidates]
        scores = self.cross_encoder.predict(pairs)
        reranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
        
        # Return top-k
        return [doc for _, doc in reranked[:k_final]]

retriever = MultiStageRetriever(
    hybrid_retriever,
    'cross-encoder/ms-marco-MiniLM-L-6-v2'
)

# Use in RAG chain
query = "What is deep learning?"
context_docs = retriever.retrieve(query, k_final=3)
print(f"Retrieved {len(context_docs)} documents for generation.")
```

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is the advantage of hybrid search over dense-only retrieval?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387400001" value="0">
      <span>Faster inference speed</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387400001" value="1">
      <span>Combines semantic and lexical matching for robust retrieval</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387400001" value="2">
      <span>Reduces model size</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387400001" value="3">
      <span>Improves tokenization</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ When should cross-encoder reranking be used?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387400002" value="0">
      <span>For initial retrieval of all documents</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387400002" value="1">
      <span>For embedding generation</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387400002" value="2">
      <span>For reranking top-k candidates from initial retrieval</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387400002" value="3">
      <span>For tokenization</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>